## Synthetically generate humanised offensive speech detection dataset for evaluation

To run, activate the `openai_environment` conda environment specified in the environements directory. Furthermore, replace the API_KEY placeholder with a real OpenAI API key.

In [ ]:
from openai import OpenAI
import json

# instantiate api connection
client = OpenAI(
  api_key="API-KEY"
)

output_total = 300
batch_size = 10
iteration_total = output_total//batch_size

# tracks the comments that have been 
# previously generated
seen = set()
dataset = []

# helper function to generate a batch
# of comments 
def generate_comments(past_comments):
    last_50_comments = past_comments[-50:]
    raw_output = client.responses.create(
        model="gpt-5.2",
        input = [
            {
                "role": "system",
                "content": ( "You are a data generator. You generate synthetic feedback comments. Return only valid JSON with no extra text. Each object schema: [{\"comment\": \"...\", \"label\": 0 or 1}, {\"comment\": \"...\", \"label\": 0 or 1}, ...]. All generated comments must be UNIQUE." 
                )
            },
            {
                "role": "user",
                "content": (
                    f"Generate exactly {batch_size} unique student comments and corresponding labels of whether they contain offensive speech. 0 = does not contain offensive speech and 1 = contains offensive speech. There should be equal number of comments containing offensive speech and not containing offensive speech. At least 25% of comments should have 2-3 sentences.  Comments MUST have spelling mistakes, grammar errors and abbreviations. Do not repeat comments already in this list: " + 
                    "\n".join([f"-{comment}" for comment in last_50_comments])
                )
            }
        ],
        store=True,
    ).output_text
    
    processed_json = json.loads(raw_output)
    return processed_json

# generate all batches of comments
for i in range(iteration_total):
    past_comments = [entry["comment"] for entry in dataset]
    new_comments = generate_comments(past_comments)

    new_entries = []
    for new_comment in new_comments:
        c = new_comment["comment"]
        if c not in seen:
            seen.add(c)
            new_entries.append(new_comment)

    dataset.extend(new_entries)


# save comments produced
with open("dataset/test.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
